### Dataset: Energy Efficiency with HL (heating load) as target

In [1]:
from ucimlrepo import fetch_ucirepo

energy_efficiency = fetch_ucirepo(id=242) 
  
X = energy_efficiency.data.features 
y = energy_efficiency.data.targets

var_df = energy_efficiency.variables
col_map = dict(zip(var_df["name"], var_df["description"]))
X = X.rename(columns=col_map)
y = y.rename(columns=col_map)

y = y[["Heating Load"]]
df = X.join(y)
df.head()

,Relative Compactness,Surface Area,Wall Area,Roof Area,Overall Height,Orientation,Glazing Area,Glazing Area Distribution,Heating Load
0,0.98,514.5,294.0,110.25,7.0,2,0.0,0,15.55
1,0.98,514.5,294.0,110.25,7.0,3,0.0,0,15.55
2,0.98,514.5,294.0,110.25,7.0,4,0.0,0,15.55
3,0.98,514.5,294.0,110.25,7.0,5,0.0,0,15.55
4,0.90,563.5,318.5,122.50,7.0,2,0.0,0,20.84


### Train test split

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.1,     
    random_state=42,   
    shuffle=True
)

print(f"#X_train = {len(X_train)}   #y_train = {len(y_train)}")
print(f"#X_test = {len(X_test)}     #y_test = {len(y_test)}")

#X_train = 691   #y_train = 691
#X_test = 77     #y_test = 77


### EDF normalization

In [3]:
from src.edf import edf_normalize, col_denorm

X_train_norm, X_test_norm, _ = edf_normalize(X_train, X_test)
y_train_norm, y_test_norm, edf_models = edf_normalize(y_train, y_test)

y_denorm = col_denorm("Heating Load", edf_models)

### Basic evaluation for different features and target degrees

In [4]:
from src.features import moment_like_features, prepare_targets
from src.hcr import fit_lasso, plot_example_densities
from src.evaluation import mean_log_likelihood, mse_evaluation
from collections import defaultdict

N_vals = [n for n in range(2, 9)]
A = 3.
B=100.

ll_softplus_vals = defaultdict(dict)
ll_param_softplus_vals = defaultdict(dict)
ll_clip_vals = defaultdict(dict)

MSE_softplus_vals = defaultdict(dict)
MSE_param_softplus_vals = defaultdict(dict)
MSE_clip_vals = defaultdict(dict)

for n_feature in N_vals:
    for n_target in N_vals:
        print(f"----- Target deg. {n_target}, feature deg. {n_feature} -----")

        V_train = moment_like_features(X_train_norm, n_feature)
        V_test  = moment_like_features(X_test_norm, n_feature)

        targets_train = prepare_targets(y_train_norm, n_target)
        targets_test  = prepare_targets(y_test_norm, n_target)

        models = []
        for n in range(n_target):
            models.append(fit_lasso(V_train, targets_train[n]))

        plot_example_densities(V_test, y_test_norm, models,
                               name=f"Example densities (\"softplus\", features deg. {n_feature}, target deg. {n_target})",
                               method="softplus", seed=42, save=True)
        plot_example_densities(V_test, y_test_norm, models,
                               name=f"Example densities (\"param-softplus\", features deg. {n_feature}, target deg. {n_target})",
                               method="softplus", 
                               a=A, b=B, seed=42, save=True)
        plot_example_densities(V_test, y_test_norm, models,
                               name=f"Example densities (\"clip\", features deg. {n_feature}, target deg. {n_target})",
                               method="clip", seed=42, save=True)
        
        ll_softplus = mean_log_likelihood(V_test, y_test_norm, models)
        ll_param_softplus = mean_log_likelihood(V_test, y_test_norm, models,
                                                method="softplus", a=A, b=B)
        ll_clip = mean_log_likelihood(V_test, y_test_norm, models, method="clip")

        mse_softplus = mse_evaluation(V_test, y_test, models, y_denorm)
        mse_param_softplus = mse_evaluation(V_test, y_test, models, y_denorm,
                                            method="softplus", a=A, b=B)
        mse_clip = mse_evaluation(V_test, y_test, models, y_denorm, method="clip")
        
        ll_softplus_vals[n_target][n_feature] = ll_softplus
        ll_param_softplus_vals[n_target][n_feature] = ll_param_softplus
        ll_clip_vals[n_target][n_feature] = ll_clip

        MSE_softplus_vals[n_target][n_feature] = mse_softplus
        MSE_param_softplus_vals[n_target][n_feature] = mse_param_softplus
        MSE_clip_vals[n_target][n_feature] = mse_clip

----- Target deg. 2, feature deg. 2 -----
----- Target deg. 3, feature deg. 2 -----
----- Target deg. 4, feature deg. 2 -----
----- Target deg. 5, feature deg. 2 -----
----- Target deg. 6, feature deg. 2 -----
----- Target deg. 7, feature deg. 2 -----
----- Target deg. 8, feature deg. 2 -----
----- Target deg. 2, feature deg. 3 -----
----- Target deg. 3, feature deg. 3 -----
----- Target deg. 4, feature deg. 3 -----
----- Target deg. 5, feature deg. 3 -----
----- Target deg. 6, feature deg. 3 -----
----- Target deg. 7, feature deg. 3 -----
----- Target deg. 8, feature deg. 3 -----
----- Target deg. 2, feature deg. 4 -----
----- Target deg. 3, feature deg. 4 -----
----- Target deg. 4, feature deg. 4 -----
----- Target deg. 5, feature deg. 4 -----
----- Target deg. 6, feature deg. 4 -----
----- Target deg. 7, feature deg. 4 -----
----- Target deg. 8, feature deg. 4 -----
----- Target deg. 2, feature deg. 5 -----
----- Target deg. 3, feature deg. 5 -----
----- Target deg. 4, feature deg. 

In [14]:
for n_feature in N_vals:
    for n_target in N_vals:
        print(f"Features deg. {n_feature}, Target deg. {n_target}:")

        print(f"  Mean log-likelihood (softplus):       {ll_softplus_vals[n_target][n_feature]:.4f}")
        print(f"  Mean log-likelihood (param-softplus): {ll_param_softplus_vals[n_target][n_feature]:.4f}")
        print(f"  Mean log-likelihood (clip):           {ll_clip_vals[n_target][n_feature]:.4f}")

        print(f"  MSE (softplus):                       {MSE_softplus_vals[n_target][n_feature]:.4f}")
        print(f"  MSE (param-softplus):                 {MSE_param_softplus_vals[n_target][n_feature]:.4f}")
        print(f"  MSE (clip):                           {MSE_clip_vals[n_target][n_feature]:.4f}")

Features deg. 2, Target deg. 2:
  Mean log-likelihood (softplus):       0.4933
  Mean log-likelihood (param-softplus): 0.8070
  Mean log-likelihood (clip):           0.6838
  MSE (softplus):                       28.4662
  MSE (param-softplus):                 11.6983
  MSE (clip):                           14.9911
Features deg. 2, Target deg. 3:
  Mean log-likelihood (softplus):       0.6033
  Mean log-likelihood (param-softplus): 0.9390
  Mean log-likelihood (clip):           0.8241
  MSE (softplus):                       27.7673
  MSE (param-softplus):                 8.9418
  MSE (clip):                           13.5442
Features deg. 2, Target deg. 4:
  Mean log-likelihood (softplus):       0.6548
  Mean log-likelihood (param-softplus): 0.9899
  Mean log-likelihood (clip):           0.8659
  MSE (softplus):                       28.2820
  MSE (param-softplus):                 9.0898
  MSE (clip):                           14.5121
Features deg. 2, Target deg. 5:
  Mean log-likeliho

In [6]:
def process_results(results_dict, reverse=True):
    sorted_items = sorted(
        ((n_target, n_features, val) for n_target, inner in results_dict.items() for n_features, val in inner.items()),
        key=lambda x: x[2],
        reverse=reverse
    )
    return sorted_items

In [7]:
def print_best_results(results_dict, results_type="", cal_method="", N_vals=10, reverse=True):
    sorted = process_results(results_dict, reverse=reverse)
    print(f"Best {results_type} values for \"{cal_method}\" calibration:")
    for id in range(N_vals):
        target_deg, feature_deg, val = sorted[id]
        print(f"{id+1}. Target deg: {target_deg}, feature deg: {feature_deg}")
        print(f"    {val:.4f}")

In [8]:
print_best_results(ll_softplus_vals, 
                   results_type="mean log-likelihood",
                   cal_method="softplus")

Best mean log-likelihood values for "softplus" calibration:
1. Target deg: 8, feature deg: 8
    0.9119
2. Target deg: 8, feature deg: 7
    0.9103
3. Target deg: 7, feature deg: 8
    0.9093
4. Target deg: 7, feature deg: 7
    0.9076
5. Target deg: 8, feature deg: 6
    0.9042
6. Target deg: 8, feature deg: 5
    0.9022
7. Target deg: 7, feature deg: 6
    0.9014
8. Target deg: 7, feature deg: 5
    0.8993
9. Target deg: 8, feature deg: 4
    0.8881
10. Target deg: 7, feature deg: 4
    0.8853


In [9]:
print_best_results(MSE_softplus_vals, 
                   results_type="mean squared error",
                   cal_method="softplus",
                   reverse=False)

Best mean squared error values for "softplus" calibration:
1. Target deg: 5, feature deg: 6
    22.6966
2. Target deg: 5, feature deg: 5
    22.7153
3. Target deg: 5, feature deg: 7
    22.7242
4. Target deg: 5, feature deg: 8
    22.7460
5. Target deg: 3, feature deg: 8
    22.9294
6. Target deg: 3, feature deg: 7
    22.9512
7. Target deg: 3, feature deg: 6
    22.9684
8. Target deg: 3, feature deg: 5
    22.9791
9. Target deg: 4, feature deg: 6
    22.9864
10. Target deg: 4, feature deg: 5
    23.0174


In [10]:
print_best_results(ll_param_softplus_vals, 
                   results_type="mean log-likelihood",
                   cal_method="param-softplus")

Best mean log-likelihood values for "param-softplus" calibration:
1. Target deg: 7, feature deg: 8
    1.3835
2. Target deg: 7, feature deg: 7
    1.3821
3. Target deg: 7, feature deg: 6
    1.3753
4. Target deg: 7, feature deg: 5
    1.3741
5. Target deg: 7, feature deg: 4
    1.3667
6. Target deg: 8, feature deg: 8
    1.3577
7. Target deg: 8, feature deg: 7
    1.3566
8. Target deg: 8, feature deg: 6
    1.3504
9. Target deg: 8, feature deg: 5
    1.3495
10. Target deg: 8, feature deg: 4
    1.3464


In [11]:
print_best_results(MSE_param_softplus_vals, 
                   results_type="mean squared error",
                   cal_method="param-softplus",
                   reverse=False)

Best mean squared error values for "param-softplus" calibration:
1. Target deg: 8, feature deg: 4
    3.2055
2. Target deg: 7, feature deg: 5
    3.2836
3. Target deg: 7, feature deg: 6
    3.2955
4. Target deg: 8, feature deg: 5
    3.3259
5. Target deg: 8, feature deg: 6
    3.3330
6. Target deg: 7, feature deg: 4
    3.3546
7. Target deg: 8, feature deg: 7
    3.3993
8. Target deg: 8, feature deg: 8
    3.4291
9. Target deg: 7, feature deg: 7
    3.4389
10. Target deg: 7, feature deg: 8
    3.4672


In [12]:
print_best_results(ll_clip_vals, 
                   results_type="mean log-likelihood",
                   cal_method="clip")

Best mean log-likelihood values for "clip" calibration:
1. Target deg: 7, feature deg: 8
    1.1446
2. Target deg: 7, feature deg: 7
    1.1425
3. Target deg: 8, feature deg: 8
    1.1393
4. Target deg: 8, feature deg: 7
    1.1373
5. Target deg: 7, feature deg: 6
    1.1348
6. Target deg: 7, feature deg: 5
    1.1326
7. Target deg: 8, feature deg: 6
    1.1298
8. Target deg: 8, feature deg: 5
    1.1278
9. Target deg: 7, feature deg: 4
    1.1176
10. Target deg: 8, feature deg: 4
    1.1123


In [13]:
print_best_results(MSE_clip_vals, 
                   results_type="mean squared error",
                   cal_method="clip",
                   reverse=False)

Best mean squared error values for "clip" calibration:
1. Target deg: 4, feature deg: 6
    7.9255
2. Target deg: 4, feature deg: 8
    7.9269
3. Target deg: 4, feature deg: 5
    7.9451
4. Target deg: 4, feature deg: 7
    7.9599
5. Target deg: 3, feature deg: 8
    8.0152
6. Target deg: 3, feature deg: 7
    8.0240
7. Target deg: 4, feature deg: 4
    8.0609
8. Target deg: 3, feature deg: 5
    8.0789
9. Target deg: 3, feature deg: 6
    8.1102
10. Target deg: 5, feature deg: 7
    8.4994


### CV for top basic results

In [19]:
from src.cv import cross_validate, print_cv_results

def cv_top(X, y, results_dict, 
           results_type, method,
           a=1, b=1, eps=1e-6,
           N_vals=5, reverse=True):
    sorted = process_results(results_dict, reverse=reverse)
    results = []
    print(f"CV for top {results_type} values with \"{method}\" calibration:")
    if method=="param-softplus": method="softplus"

    for id in range(N_vals):
        target_deg, feature_deg, _ = sorted[id]
        print(f"{id+1}. Target deg: {target_deg}, feature deg: {feature_deg}")
        ll_results, mse_results = cross_validate(X, y,
                                                 feature_deg, target_deg,
                                                 method=method,
                                                 a=a, b=b, eps=eps)
        results.append({"target_deg": target_deg,
                        "feature_deg": feature_deg,
                        "ll_results": ll_results,
                        "mse_results": mse_results})
        print_cv_results(ll_results, mse_results)
    return results

In [20]:
cv_ll_softplus_results = cv_top(X, y, ll_softplus_vals,
                                results_type="mean log-likelihood",
                                method="softplus")

CV for top mean log-likelihood values with "softplus" calibration:
1. Target deg: 8, feature deg: 8
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [0.84 0.88 0.92 0.81 0.9  0.92 0.85 0.83 0.83 0.81]
  mean LL : 0.8596
  std LL  : 0.0406

Mean square error:
  per fold: [18.17 18.44 17.22 22.4  27.88 16.02 16.96 24.36 18.08 17.25]
  mean MSE: 19.6778
  std MSE : 3.6824
2. Target deg: 8, feature deg: 7
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [0.84 0.88 0.92 0.81 0.9  0.91 0.85 0.83 0.83 0.81]
  mean LL : 0.8589
  std LL  : 0.0402

Mean square error:
  per fold: [18.2  18.35 17.23 22.28 27.78 16.04 16.89 24.37 18.13 17.29]
  mean MSE: 19.6543
  std MSE : 3.6527
3. Target deg: 7, feature deg: 8
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [0.83 0.89 0.93 0.82 0.89 0.91 0.81 0.82 0.82 0.8 ]
  mean LL : 0.8519
  std LL  : 0.043

In [21]:
cv_mse_softplus_results = cv_top(X, y, MSE_softplus_vals,
                                 results_type="mean squared error",
                                 method="softplus",
                                 reverse=False)

CV for top mean squared error values with "softplus" calibration:
1. Target deg: 5, feature deg: 6
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [0.77 0.78 0.81 0.77 0.84 0.82 0.73 0.76 0.74 0.71]
  mean LL : 0.7719
  std LL  : 0.0391

Mean square error:
  per fold: [17.91 17.76 17.14 21.69 27.11 15.78 16.69 24.08 17.97 17.31]
  mean MSE: 19.3441
  std MSE : 3.5140
2. Target deg: 5, feature deg: 5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [0.77 0.77 0.81 0.77 0.84 0.82 0.73 0.76 0.74 0.71]
  mean LL : 0.7712
  std LL  : 0.0390

Mean square error:
  per fold: [17.91 17.77 17.14 21.66 27.14 15.78 16.68 24.07 17.96 17.31]
  mean MSE: 19.3434
  std MSE : 3.5181
3. Target deg: 5, feature deg: 7
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [0.77 0.78 0.81 0.77 0.84 0.82 0.73 0.76 0.74 0.72]
  mean LL : 0.7748
  std LL  : 0.0392

In [23]:
cv_ll_param_softplus_results = cv_top(X, y, ll_param_softplus_vals,
                                      results_type="mean log-likelihood",
                                      method="param-softplus",
                                      a=3., b=100.)

CV for top mean log-likelihood values with "param-softplus" calibration:
1. Target deg: 7, feature deg: 8
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [1.13 1.29 1.48 1.1  1.25 1.34 1.05 1.2  1.07 1.17]
  mean LL : 1.2094
  std LL  : 0.1261

Mean square error:
  per fold: [3.59 2.78 2.15 3.41 5.46 3.   4.58 4.09 5.17 2.45]
  mean MSE: 3.6669
  std MSE : 1.0763
2. Target deg: 7, feature deg: 7
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [1.13 1.29 1.48 1.1  1.26 1.34 1.06 1.2  1.08 1.17]
  mean LL : 1.2098
  std LL  : 0.1252

Mean square error:
  per fold: [3.55 2.71 2.15 3.3  5.45 3.01 4.56 4.1  5.12 2.46]
  mean MSE: 3.6410
  std MSE : 1.0719
3. Target deg: 7, feature deg: 6
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [1.12 1.29 1.48 1.11 1.26 1.33 1.06 1.2  1.08 1.17]
  mean LL : 1.2090
  std LL  : 0.1258

Mean square e

In [24]:
cv_mse_param_softplus_results = cv_top(X, y, MSE_param_softplus_vals,
                                       results_type="mean squared error",
                                       method="param-softplus",
                                       a=3., b=100.,
                                       reverse=False)

CV for top mean squared error values with "param-softplus" calibration:
1. Target deg: 8, feature deg: 4
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [1.12 1.26 1.45 1.07 1.29 1.3  1.16 1.2  1.1  1.14]
  mean LL : 1.2074
  std LL  : 0.1107

Mean square error:
  per fold: [3.32 2.92 2.4  2.91 5.18 3.63 4.5  4.06 4.78 2.79]
  mean MSE: 3.6478
  std MSE : 0.8925
2. Target deg: 7, feature deg: 5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [1.12 1.29 1.48 1.11 1.26 1.33 1.06 1.2  1.08 1.17]
  mean LL : 1.2083
  std LL  : 0.1254

Mean square error:
  per fold: [3.46 2.7  2.1  3.19 5.36 3.01 4.54 4.03 5.03 2.54]
  mean MSE: 3.5955
  std MSE : 1.0464
3. Target deg: 7, feature deg: 6
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [1.12 1.29 1.48 1.11 1.26 1.33 1.06 1.2  1.08 1.17]
  mean LL : 1.2090
  std LL  : 0.1258

Mean square er

In [25]:
cv_ll_clip_results = cv_top(X, y, ll_clip_vals,
                            results_type="mean log-likelihood",
                            method="clip")

CV for top mean log-likelihood values with "clip" calibration:
1. Target deg: 7, feature deg: 8
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [1.04 1.07 1.17 1.02 0.96 1.14 1.01 1.04 1.03 0.87]
  mean LL : 1.0346
  std LL  : 0.0799

Mean square error:
  per fold: [ 9.12  8.53  7.82 10.79 15.48  7.75  7.98 12.68  8.91  8.19]
  mean MSE: 9.7258
  std MSE : 2.4161
2. Target deg: 7, feature deg: 7
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [1.04 1.07 1.17 1.02 0.96 1.13 1.01 1.04 1.03 0.87]
  mean LL : 1.0338
  std LL  : 0.0795

Mean square error:
  per fold: [ 9.09  8.55  7.86 10.76 15.46  7.78  7.98 12.7   8.94  8.17]
  mean MSE: 9.7298
  std MSE : 2.4065
3. Target deg: 8, feature deg: 8
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [1.06 0.95 1.16 0.86 0.97 1.14 1.06 1.03 1.03 0.88]
  mean LL : 1.0134
  std LL  : 0.0951

Mea

In [26]:
cv_mse_clip_results = cv_top(X, y, MSE_clip_vals,
                             results_type="mean squared error",
                             method="clip",
                             reverse=False)

CV for top mean squared error values with "clip" calibration:
1. Target deg: 4, feature deg: 6
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [0.95 0.89 0.99 0.92 0.84 1.   0.88 0.93 0.93 0.72]
  mean LL : 0.9051
  std LL  : 0.0770

Mean square error:
  per fold: [ 7.17  5.52  5.99  7.27 10.84  6.36  5.32  7.59  7.11  5.24]
  mean MSE: 6.8396
  std MSE : 1.5654
2. Target deg: 4, feature deg: 8
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [0.95 0.77 0.99 0.91 0.84 1.01 0.89 0.93 0.93 0.72]
  mean LL : 0.8946
  std LL  : 0.0880

Mean square error:
  per fold: [ 6.9   5.46  5.84  7.28 10.97  6.28  5.13  7.64  7.07  5.02]
  mean MSE: 6.7593
  std MSE : 1.6520
3. Target deg: 4, feature deg: 5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [0.94 0.89 0.99 0.93 0.84 1.   0.88 0.93 0.93 0.72]
  mean LL : 0.9049
  std LL  : 0.0769

Mean

### Results summary

In [27]:
import pandas as pd
import numpy as np

def results_to_df(results):
    rows = []
    for r in results:
        rows.append({
            "target_deg": r["target_deg"],
            "feature_deg": r["feature_deg"],
            "ll_mean": np.mean(r["ll_results"]),
            "ll_std": np.std(r["ll_results"]),
            "mse_mean": np.mean(r["mse_results"]),
            "mse_std": np.std(r["mse_results"]),
        })
    return pd.DataFrame(rows)

In [28]:
results = {
    "ll_softplus": cv_ll_softplus_results,
    "mse_softplus": cv_mse_softplus_results,

    "ll_param_softplus": cv_ll_param_softplus_results,
    "mse_param_softplus": cv_mse_param_softplus_results,

    "ll_clip": cv_ll_clip_results,
    "mse_clip": cv_mse_clip_results,
}

In [29]:
results_dfs = {key : results_to_df(val) for key, val in results.items()}

In [30]:
df = results_dfs["ll_softplus"]
df

,target_deg,feature_deg,ll_mean,ll_std,mse_mean,mse_std
0,8,8,0.859608,0.040608,19.677821,3.682443
1,8,7,0.858929,0.040220,19.654283,3.652677
2,7,8,0.851872,0.043531,19.582961,3.773343
3,7,7,0.851191,0.043141,19.565382,3.759285
4,8,6,0.855889,0.040073,19.651283,3.623018


In [31]:
df = results_dfs["mse_softplus"]
df

,target_deg,feature_deg,ll_mean,ll_std,mse_mean,mse_std
0,5,6,0.771922,0.039127,19.344125,3.514035
1,5,5,0.771206,0.039048,19.343393,3.518071
2,5,7,0.774832,0.039226,19.295822,3.588557
3,5,8,0.775392,0.039431,19.269474,3.611001
4,3,8,0.639899,0.043827,19.126406,3.743433


In [32]:
df = results_dfs["ll_param_softplus"]
df

,target_deg,feature_deg,ll_mean,ll_std,mse_mean,mse_std
0,7,8,1.209406,0.126062,3.666906,1.076268
1,7,7,1.209785,0.125244,3.640976,1.071912
2,7,6,1.208972,0.125765,3.617753,1.060907
3,7,5,1.208302,0.125436,3.595493,1.046377
4,7,4,1.199650,0.124548,3.636729,0.953397


In [33]:
df = results_dfs["mse_param_softplus"]
df

,target_deg,feature_deg,ll_mean,ll_std,mse_mean,mse_std
0,8,4,1.207419,0.110687,3.647847,0.892528
1,7,5,1.208302,0.125436,3.595493,1.046377
2,7,6,1.208972,0.125765,3.617753,1.060907
3,8,5,1.214203,0.113045,3.606554,0.966998
4,8,6,1.214510,0.113796,3.626501,0.978138


In [34]:
df = results_dfs["ll_clip"]
df

,target_deg,feature_deg,ll_mean,ll_std,mse_mean,mse_std
0,7,8,1.034576,0.079938,9.725834,2.416137
1,7,7,1.033846,0.079524,9.729765,2.406462
2,8,8,1.013393,0.095148,9.731790,2.358616
3,8,7,1.012542,0.094647,9.719591,2.360885
4,7,6,1.031765,0.079825,9.749875,2.407678


In [35]:
df = results_dfs["mse_clip"]
df

,target_deg,feature_deg,ll_mean,ll_std,mse_mean,mse_std
0,4,6,0.905113,0.077009,6.839573,1.565369
1,4,8,0.894567,0.087980,6.759264,1.651998
2,4,5,0.904947,0.076927,6.794660,1.532180
3,4,7,0.903554,0.077943,6.776647,1.630622
4,3,8,0.760975,0.193556,7.078725,1.903237
